# DenseNet & EfficientNet — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def conv2d(x,k,pad=0,stride=1,dilation=1):
    x=np.asarray(x,float); k=np.asarray(k,float)
    if pad: x=np.pad(x,pad)
    kh,kw=k.shape; dh=(kh-1)*dilation+1; dw=(kw-1)*dilation+1; out=[]
    for i in range(0,x.shape[0]-dh+1,stride):
        row=[]
        for j in range(0,x.shape[1]-dw+1,stride): row.append(float(np.sum(x[i:i+dh:dilation,j:j+dw:dilation]*k)))
        out.append(row)
    return np.array(out)
def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); union=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union else 0
def softmax(z):
    z=np.asarray(z,float); e=np.exp(z-z.max()); return e/e.sum()
def show(a,title,cmap='viridis'):
    plt.figure(figsize=(4,3)); plt.imshow(a,cmap=cmap); plt.colorbar(); plt.title(title)
print('setup ok')

## DenseNet & EfficientNet
Tiny CPU-only synthetic arrays for DenseNet & EfficientNet: no downloads, no pretrained models, and every cell ends with an assert.

In [ ]:
# 1) local dot product
patch=np.array([[1,2],[3,4]]); k=np.array([[1,0],[0,-1]]); prod=patch*k; show(prod,'elementwise products')
assert prod.sum()==-3

In [ ]:
# 2) sliding convolution map
img=np.arange(1,10).reshape(3,3); y=conv2d(img,np.array([[1,0],[0,-1]])); show(y,'feature map')
assert y.shape==(2,2) and np.all(y==-4)

In [ ]:
# 3) stride and padding change shape
img=np.arange(25).reshape(5,5); k=np.ones((2,2)); a=conv2d(img,k,stride=2); b=conv2d(img,k,pad=1); show(a,'stride-2 map')
assert a.shape==(2,2) and b.shape==(6,6)

In [ ]:
# 4) max pooling preserves strongest activation
x=np.array([[1,3,2,0],[4,6,5,1],[1,2,9,8],[0,1,7,3]]); y=np.array([[x[i:i+2,j:j+2].max() for j in range(0,4,2)] for i in range(0,4,2)]); show(y,'max pool')
assert np.array_equal(y,[[6,5],[2,9]])

In [ ]:
# 5) channel mixing and residual addition
x=np.array([1.,2.,3.]); F=np.array([.1,-.2,.3]); y=x+F; plt.figure(figsize=(4,3)); plt.plot(x,'o-',label='x'); plt.plot(y,'s-',label='x+F'); plt.legend(); plt.title('residual correction')
assert np.allclose(y,[1.1,1.8,3.3])